<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/M0_2048_FEATURE_COMPATIBILITY_v1_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M0_2048_FEATURE_COMPATIBILITY_v1

**승인**: HoonNi / 2026-10-24 / 지시서 15 (코드 원안은 HoonNi 제공, 설정값만 에이전트가 채움)

**목적**: FULL_CONVERT_v1_2 산출물(2048@20.48kHz)이 동결 M0(2560@25.6kHz) baseline과 9특징·검출 결과 수준에서
호환되는지 확인한다(D-021 기준: Spearman≥0.95, onset 차이≤5윈도).

## 동결 함수 식별 결과 (에이전트, 읽기 전용 확인)

- 동결 노트북: `field_iis3dwb/M0_FEMTO_Baseline_v1.ipynb` (로컬 SHA-256: `921F1BBEA7B460D748C6DC3F74C04981F3BF4DE5FF76223B188E759CC7E6EE60`)
- 9특징 함수: **`extract_features(sig, fs=FEMTO_FS)`** — RMS/Peak/Crest/Kurtosis/Skew/P2P/Std/EnvRMS/HF_ratio 9D 반환
- 판정 함수: **`register_and_detect(feats, k=K, consec=CONSEC, reg_min=REG_MIN, reg_max=REG_MAX)`** — 등록+Mahalanobis+5연속 이탈 판정. 반환 dict: `n_reg, thr, dists, alarm_idx, N, reg_far`
- 전역 상수(동결 노트북 Cell A, 대문자 변수라 Cell 2 추출 로직이 자동으로 가져옴 — FROZEN_EXTRA_GLOBALS 불필요): `K=6.0, CONSEC=5, REG_MIN=90, REG_MAX=600, FEMTO_FS=25600, COL_H=4, FEMTO_INTERVAL_SEC=10, FEATURE_NAMES=[...]`

## [확인 필요] 항목 (대표 판단 필요, 임의로 고치지 않음)

1. **REF_COLS["detected"] 매핑 문제**: `m0_baseline_result.csv`의 실제 열은
   `bearing, N, n_reg, alarm_idx, life_pct, lead_frames, lead_hours, reg_FAR, reg_ok, status` 이다.
   "detected" 역할을 할 만한 열은 `status`(값: `"✅ 탐지"` / `"❌ 미탐지"`)뿐인데,
   Cell 3의 `_bool()` 헬퍼는 `("true","1","yes","y","detected")` 문자열만 True로 인식해
   한글 체크마크 텍스트("✅ 탐지")는 **False로 오판정될 위험**이 있다.
   이 노트북은 `_bool()` 로직을 바꾸지 않고(지시사항 "로직 변경 금지") `REF_COLS["detected"]="status"`로 채웠다.
   **Cell 3 실행 결과에서 모든 베어링이 ref_detected=False로 나오면 이 문제가 현실화된 것** — 그 경우
   `_bool()` 수정 또는 `alarm_idx`의 notna 여부로 대체할지 대표 판단이 필요하다.
2. **lead_hours 산출식**: `m0_detect` 어댑터에서 `lead_hours = (N - alarm_idx) * FEMTO_INTERVAL_SEC / 3600`를 계산한다.
   이 식은 동결 노트북 Cell C에 있는 동일한 공식(`lead_f = N - alarm_idx; lead_sec = lead_f*FEMTO_INTERVAL_SEC; lead_h = lead_sec/3600`)을
   **그대로 복사**한 것이며 새로운 계산식을 추가한 것이 아니다. 다만 "계산식 추가 금지" 원칙과 경계에 있는 판단이라 대표 확인을 요청한다.
3. **FROZEN_NOTEBOOK_PATH**: 로컬 workspace 사본 경로(field_iis3dwb/M0_FEMTO_Baseline_v1.ipynb)를 기준으로
   Colab 절대경로 /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/M0_FEMTO_Baseline_v1.ipynb 를 채웠다(동결 노트북 Cell A의
   SAVE_DIR = DRIVE + '/Colab Notebooks/field_iis3dwb/' 와 일치). Drive 실제 배치가 다르면 Cell 2 실행 시 후보 목록이 출력되므로 그때 교정 가능.

## 실행 순서

Cell 1(Gate) → Cell 2(동결 함수 추출) → Cell 2B(어댑터) → Cell 3(원본 쪽 + 재현 게이트, 1.5~2시간) → Cell 4(2048 쪽, 수분) → Cell 5(비교+판정) → Cell 6(최종 요약)

## 절대 원칙

- 원본 CSV·동결 파일·FEMTO_2048_v1_2 데이터셋에 쓰지 않는다(읽기 전용)
- 판정 기준(Spearman≥0.95, onset차이≤5윈도)은 노트북 안에서 변경 불가
- M0 재구현 금지 — 동결 함수를 그대로 가져와 실행만 한다

In [ ]:
# ================================================================
# Cell 1 — 설정 + Gate (FULL_CONVERT_PASS 확인, 동결 산출물 위치·해시 기록)
# 읽기 전용: 원본 CSV·동결 파일·FEMTO_2048_v1_2 데이터셋에 쓰지 않는다.
# ================================================================
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import ast, builtins, dis, hashlib, inspect, io, json, os, re, sys, time
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import scipy
from scipy.stats import spearmanr

VERSION = "M0_2048_FEATURE_COMPATIBILITY_v1"
PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/field_iis3dwb")
BRIDGE_ROOT = PROJECT_ROOT / "bridge_outputs"
DATASET_ROOT = PROJECT_ROOT / "bridge_datasets/FEMTO_2048_v1_2"
CACHE_DIR = BRIDGE_ROOT / "_cache/M0_2048_COMPAT_v1"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_m0_2048_compat_v1")
REPORT_DIR = BRIDGE_ROOT / RUN_ID
REPORT_DIR.mkdir(parents=True, exist_ok=False)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

LEARNING_BEARINGS = ["Bearing1_1", "Bearing1_2", "Bearing2_1",
    "Bearing2_2", "Bearing3_1", "Bearing3_2"]
SOURCE_FS, SOURCE_SAMPLES = 25600, 2560
OUTPUT_FS, OUTPUT_SAMPLES = 20480, 2048
COL_H, COL_COUNT_REQUIRED = 4, 6
DECISION_INTERVAL_SEC = 10.0  # N-7: FEMTO 원래 간격(10초 스냅샷)

# 합의 기준 (HoonNi 동의 2026-10-22) — 이 노트북 안에서 변경 금지
CRITERIA = {
    "feature_spearman_min": 0.95,  # 9특징 × 6베어링 54개 모두
    "onset_diff_max_windows": 5,   # |onset_2048 - onset_원본| <= 5
    "detected_set_must_match": True,  # 검출 베어링 집합 동일
}
CRITERIA_SHA256 = hashlib.sha256(json.dumps(CRITERIA, sort_keys=True).encode()).hexdigest()

def sha256_file(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def write_json(path, obj):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    os.replace(tmp, path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

IO_RETRY_WAITS = [2, 5, 10, 20]
def with_io_retry(fn, *args):
    last = None
    for i in range(len(IO_RETRY_WAITS) + 1):
        try:
            return fn(*args)
        except OSError as e:
            last = e
            if i < len(IO_RETRY_WAITS):
                time.sleep(IO_RETRY_WAITS[i])
    raise last

def stop(status, reason, **extra):
    write_json(REPORT_DIR / "COMPAT_FINAL_SUMMARY.json", {
        "version": VERSION, "status": status, "reason": reason,
        "created_at": datetime.now().isoformat(timespec="seconds"), **extra})
    raise RuntimeError(f"{status}: {reason}")

# ── Gate: 최신 FULL_CONVERT_v1_2 PASS ──
fc_dirs = []
for p in BRIDGE_ROOT.glob("*_full_convert_v1_2"):
    s = p / "FULL_CONVERT_FINAL_SUMMARY.json"
    if s.is_file() and read_json(s).get("full_convert_status") == "FULL_CONVERT_PASS":
        fc_dirs.append(p)
if not fc_dirs:
    stop("COMPAT_BLOCKED_GATE", "FULL_CONVERT_PASS 산출물 없음")
FC_DIR = sorted(fc_dirs, key=lambda p: p.stat().st_mtime_ns, reverse=True)[0]
fc_summary = read_json(FC_DIR / "FULL_CONVERT_FINAL_SUMMARY.json")
ds_contract = read_json(DATASET_ROOT / "dataset_contract.json")

dataset_manifest = pd.read_csv(DATASET_ROOT / "dataset_manifest.csv", low_memory=False)
learn_df = dataset_manifest[dataset_manifest["split"] == "LEARNING"]

gate = {
    "full_convert_pass": fc_summary.get("full_convert_status") == "FULL_CONVERT_PASS",
    "next_action_ok": fc_summary.get("recommended_next_action") == "READY_FOR_M0_2048_FEATURE_COMPATIBILITY",
    "contract_sha_match": ds_contract.get("contract_sha256") == fc_summary.get("contract_sha256"),
    "manifest_rows_39016": len(dataset_manifest) == 39016,
    "learning_rows_7534": len(learn_df) == 7534,
    "learning_bearings_6": sorted(learn_df["logical_bearing_id"].unique()) == sorted(LEARNING_BEARINGS),
}
for b in LEARNING_BEARINGS:
    gate[f"npy_exists_{b}"] = (DATASET_ROOT / "LEARNING" / b / "signals.npy").is_file()
for k, v in gate.items():
    print(f"{'PASS' if v else 'FAIL'}: {k}")
if not all(gate.values()):
    stop("COMPAT_BLOCKED_GATE", "Gate 실패", gate=gate)

# ── 동결 산출물 위치 (정확히 1개일 때만 자동 채택, 아니면 직접 지정) ──
FROZEN_JSON_PATH = None    # 예: ".../BASELINE_M0_frozen.json"
FROZEN_RESULT_CSV_PATH = None  # 예: ".../m0_baseline_result.csv"

def find_unique(name, override):
    if override:
        p = Path(override)
        if not p.is_file():
            stop("COMPAT_BLOCKED_GATE", f"지정 경로 없음: {p}")
        return p
    hits = [p for p in PROJECT_ROOT.rglob(name)
            if "md_backup" not in str(p) and "archive" not in str(p).lower()]
    if len(hits) != 1:
        stop("COMPAT_BLOCKED_GATE", f"{name} 후보 {len(hits)}개 — 경로를 직접 지정할 것",
             candidates=[str(h) for h in hits])
    return hits[0]

FROZEN_JSON = find_unique("BASELINE_M0_frozen.json", FROZEN_JSON_PATH)
FROZEN_RESULT_CSV = find_unique("m0_baseline_result.csv", FROZEN_RESULT_CSV_PATH)
FROZEN_HASH_BEFORE = {str(FROZEN_JSON): sha256_file(FROZEN_JSON),
                       str(FROZEN_RESULT_CSV): sha256_file(FROZEN_RESULT_CSV)}
print("\n동결 파일:", json.dumps(FROZEN_HASH_BEFORE, indent=2, ensure_ascii=False))
print("FULL_CONVERT:", FC_DIR.name)
print("\n[Cell 1] Gate PASS ✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PASS: full_convert_pass
PASS: next_action_ok
PASS: contract_sha_match
PASS: manifest_rows_39016
PASS: learning_rows_7534
PASS: learning_bearings_6
PASS: npy_exists_Bearing1_1
PASS: npy_exists_Bearing1_2
PASS: npy_exists_Bearing2_1
PASS: npy_exists_Bearing2_2
PASS: npy_exists_Bearing3_1
PASS: npy_exists_Bearing3_2

동결 파일: {
  "/content/drive/MyDrive/Colab Notebooks/field_iis3dwb/BASELINE_M0_frozen.json": "a4632ad1b9c519768081cea2c35238598f92144580573d65ab85343f3db608b2",
  "/content/drive/MyDrive/Colab Notebooks/field_iis3dwb/m0_baseline_result.csv": "63f493e6e3482157dba7a3b7eef75d6d679086bc2ae5b2632e06c4095b2ea4e7"
}
FULL_CONVERT: 20261024_001446_full_convert_v1_2

[Cell 1] Gate PASS ✅


In [ ]:
# ================================================================
# Cell 2 — 동결 M0 함수 추출 (r2)
# [r2 수정] ① 미해결 검사는 '동결 노트북 안에서 정의된 함수'만 재귀 (SciPy 내부 미검사)
#           ② 소스 해시는 AST 원문 기준 (exec 함수는 inspect.getsource 불가)
#           ③ 함수 호출이 포함된 대문자 대입은 실행하지 않음 (무거운 계산 차단, 목록 출력)
#           ④ 핵심 상수 값 확인 (변경 아님, 불일치 시 정지)
# ================================================================
FROZEN_NOTEBOOK_PATH = "/content/drive/MyDrive/Colab Notebooks/field_iis3dwb/M0_FEMTO_Baseline_v1.ipynb"
FROZEN_FEATURE_FN = "extract_features"
FROZEN_DETECT_FN = "register_and_detect"
FROZEN_EXTRA_GLOBALS = {}  # 미해결 전역값이 나오면 동결 노트북 원문 값 그대로만 기입

# 인수인계서 동결값 — 확인용(이 값으로 덮어쓰지 않음)
EXPECTED_FROZEN_CONSTS = {"K": 6.0, "CONSEC": 5, "REG_MIN": 90, "REG_MAX": 600,
                          "FEMTO_FS": 25600, "COL_H": 4, "FEMTO_INTERVAL_SEC": 10}

FROZEN_NB = Path(FROZEN_NOTEBOOK_PATH)
if not FROZEN_NB.is_file():
    cands = sorted(str(p) for p in PROJECT_ROOT.rglob("*.ipynb") if "m0" in p.name.lower())
    print("동결 노트북 후보:"); [print(" ", c) for c in cands]
    stop("COMPAT_BLOCKED_ADAPTER_NOT_SET", f"동결 노트북 없음: {FROZEN_NB}", candidates=cands)

FROZEN_HASH_BEFORE[str(FROZEN_NB)] = sha256_file(FROZEN_NB)
nb = json.loads(FROZEN_NB.read_text(encoding="utf-8"))

SAFE_CALLS = {"int", "float", "str", "bool", "list", "tuple", "dict", "set", "range", "len", "round", "abs"}
def _is_safe_value(v):
    for n in ast.walk(v):
        if isinstance(n, ast.Call):
            f = n.func
            if isinstance(f, ast.Name) and f.id in SAFE_CALLS: continue
            if isinstance(f, ast.Attribute) and ast.unparse(f) == "os.path.join": continue
            return False
    return True

kept, skipped_cells, skipped_assign = [], [], []
defined, def_src, const_hist = {}, {}, {}
for ci, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue
    src = cell["source"]; src = "".join(src) if isinstance(src, list) else src
    lines = []
    for l in src.splitlines():
        s = l.lstrip()
        lines.append(l[:len(l) - len(s)] + "pass" if s.startswith(("!", "%")) else l)
    try:
        tree = ast.parse("\n".join(lines))
    except SyntaxError as e:
        skipped_cells.append({"cell": ci, "error": str(e)}); continue
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            kept.append((ci, node))
            defined.setdefault(node.name, []).append(ci)
            def_src.setdefault(node.name, []).append(ast.unparse(node))
        elif isinstance(node, (ast.Import, ast.ImportFrom)):
            kept.append((ci, node))
        elif isinstance(node, (ast.Assign, ast.AnnAssign)):
            targets = node.targets if isinstance(node, ast.Assign) else [node.target]
            if node.value is None or not all(isinstance(t, ast.Name) and t.id.isupper() for t in targets):
                continue
            if _is_safe_value(node.value):
                kept.append((ci, node))
                for t in targets:
                    const_hist.setdefault(t.id, []).append((ci, ast.unparse(node.value)))
            else:
                skipped_assign.append({"cell": ci, "code": ast.unparse(node)[:120]})

FROZEN_NS = {"__name__": "frozen_m0"}
exec_fail = []
for ci, node in kept:
    try:
        exec(compile(ast.Module(body=[node], type_ignores=[]), f"<frozen cell {ci}>", "exec"), FROZEN_NS)
    except Exception as e:
        exec_fail.append({"cell": ci, "code": ast.unparse(node)[:120], "error": repr(e)})
FROZEN_NS.update(FROZEN_EXTRA_GLOBALS)

def _is_frozen_fn(o):
    return inspect.isfunction(o) and o.__globals__ is FROZEN_NS

print(f"함수/클래스 {len(defined)}개, 실행 실패 {len(exec_fail)}건, 실행 안 한 대문자 대입(함수 호출 포함) {len(skipped_assign)}건")
for name, obj in FROZEN_NS.items():
    if _is_frozen_fn(obj):
        print(f"  {name}{inspect.signature(obj)}")
if exec_fail: print("\n[실행 실패]"); display(pd.DataFrame(exec_fail))
if skipped_assign: print("\n[실행 안 함 — 함수 호출 포함]"); display(pd.DataFrame(skipped_assign))

# 중복 정의 경고 (마지막 정의 사용)
dup_fn = {k: v for k, v in defined.items() if len(v) > 1}
dup_const = {k: v for k, v in const_hist.items() if len({e for _, e in v}) > 1}
if dup_fn: print("\n⚠ 함수 중복 정의(마지막 사용):", dup_fn)
if dup_const: print("⚠ 상수 재정의(값 다름, 마지막 사용):", dup_const)
for fn in [FROZEN_FEATURE_FN, FROZEN_DETECT_FN]:
    if fn in def_src and len(set(def_src[fn])) > 1:
        print(f"⚠ {fn}: 서로 다른 정의 {len(set(def_src[fn]))}개 — 재현 게이트로 검증됨")

# 대상 함수 존재
for fn in [FROZEN_FEATURE_FN, FROZEN_DETECT_FN]:
    if not _is_frozen_fn(FROZEN_NS.get(fn)):
        stop("COMPAT_BLOCKED_ADAPTER_NOT_SET", f"동결 노트북에 함수 없음: {fn}")

# 핵심 상수 확인 (변경 아님)
const_check = {k: FROZEN_NS.get(k, "<없음>") for k in EXPECTED_FROZEN_CONSTS}
const_bad = {k: v for k, v in const_check.items() if v != EXPECTED_FROZEN_CONSTS[k]}
print("\n[동결 상수]", const_check)
if const_bad:
    stop("COMPAT_BLOCKED_FROZEN_CONST_MISMATCH", f"동결 상수 불일치: {const_bad}",
         expected=EXPECTED_FROZEN_CONSTS, found=const_check)

# 미해결 전역값 검사 — 동결 노트북 정의 함수만 재귀
def _globals_of(code):
    out = set()
    for ins in dis.get_instructions(code):
        if ins.opname in ("LOAD_GLOBAL", "LOAD_NAME"):
            out.add(ins.argval if isinstance(ins.argval, str) else str(ins.argval))
    for c in code.co_consts:
        if hasattr(c, "co_code"):
            out |= _globals_of(c)
    return out

EXTERNAL_REFS = {}
def unresolved(fn_name, seen=None):
    seen = set() if seen is None else seen
    if fn_name in seen: return set()
    seen.add(fn_name); bad = set()
    for n in _globals_of(FROZEN_NS[fn_name].__code__):
        if n in FROZEN_NS:
            o = FROZEN_NS[n]
            if _is_frozen_fn(o):
                bad |= unresolved(n, seen)
            elif callable(o) or inspect.ismodule(o):
                EXTERNAL_REFS[n] = getattr(o, "__module__", None) or getattr(o, "__name__", "")
        elif not hasattr(builtins, n):
            bad.add(n)
    return bad

_unres = sorted(unresolved(FROZEN_FEATURE_FN) | unresolved(FROZEN_DETECT_FN))
print("외부 라이브러리 참조(검사 제외):", EXTERNAL_REFS)
if _unres:
    stop("COMPAT_BLOCKED_ADAPTER_NOT_SET",
         f"동결 함수가 참조하는 미해결 전역값: {_unres} → FROZEN_EXTRA_GLOBALS에 원문 값으로 기입")

FROZEN_FN_SOURCE_SHA = {fn: hashlib.sha256(def_src[fn][-1].encode()).hexdigest()
                        for fn in [FROZEN_FEATURE_FN, FROZEN_DETECT_FN]}
print("\n[Cell 2] 동결 함수 추출 완료 ✅", FROZEN_FN_SOURCE_SHA)


함수/클래스 6개, 실행 실패 3건, 실행 안 한 대문자 대입(함수 호출 포함) 1건
  extract_features(sig, fs=25600)
  load_bearing_features(bearing_dir)
  mahal(x, mu, inv_cov)
  register_and_detect(feats, k=6.0, consec=5, reg_min=90, reg_max=600)
  classify_source(name)

[실행 실패]


,cell,code,error
0,1,FEMTO_ROOT = DRIVE + '/Colab Notebooks/3.RunTo...,"NameError(""name 'DRIVE' is not defined"")"
1,1,SAVE_DIR = DRIVE + '/Colab Notebooks/field_iis...,"NameError(""name 'DRIVE' is not defined"")"
2,3,def run_m0(femto_root=FEMTO_ROOT):\n bearin...,"NameError(""name 'FEMTO_ROOT' is not defined"")"



[실행 안 함 — 함수 호출 포함]


,cell,code
0,6,BASELINE_M0 = {'method': 'M0_9feat_Mahalanobis...



[동결 상수] {'K': 6.0, 'CONSEC': 5, 'REG_MIN': 90, 'REG_MAX': 600, 'FEMTO_FS': 25600, 'COL_H': 4, 'FEMTO_INTERVAL_SEC': 10}
외부 라이브러리 참조(검사 제외): {'hilbert': 'scipy.signal._signaltools', 'kurtosis': 'scipy.stats._stats_py', 'skew': 'scipy.stats._stats_py', 'np': 'numpy'}

[Cell 2] 동결 함수 추출 완료 ✅ {'extract_features': '41ff1876892aaeb0fde163026537906cc251e7954859dbd685ddd602dca68102', 'register_and_detect': 'f0fc94c1af7e6f69bbe9741c459bdd8effc9da1312e4a544b127d10887bd20c7'}


In [ ]:
# ================================================================
# Cell 2B — 어댑터 (에이전트 작성 구간)
# 허용: 인자 이름·순서 맞추기, 반환값 키 이름 맞추기, 타입 변환
# 금지: 계산식 추가·변경, threshold·K·CONSEC·등록 파라미터 지정/변경
# ================================================================
def m0_features(signal_1d, fs):
    fn = FROZEN_NS[FROZEN_FEATURE_FN]
    # [에이전트] extract_features(sig, fs) 시그니처와 1:1 일치 — 그대로 호출
    out = fn(signal_1d, fs)
    # [에이전트] 반환은 plain np.array(9,)이므로, 보고 편의를 위해 동결 FEATURE_NAMES로만 이름을 붙인다
    # (값 재계산 없음 — 순서 그대로 매핑, 동결 FEATURE_NAMES도 Cell 2에서 자동 추출된 원문 값)
    names = FROZEN_NS.get("FEATURE_NAMES")
    if names is not None and len(names) == len(out):
        return dict(zip(names, out))
    return out

def m0_detect(F):
    fn = FROZEN_NS[FROZEN_DETECT_FN]
    # [에이전트] register_and_detect(feats)와 1:1 일치 — k/consec/reg_min/reg_max는
    # 함수 기본값(K/CONSEC/REG_MIN/REG_MAX, 동결 노트북 원문 값)을 그대로 사용, 재지정하지 않음
    r = fn(F)
    detected = r["alarm_idx"] is not None
    # [확인 필요] lead_hours 산출: 동결 노트북 Cell C의 동일 공식을 그대로 복사한 것(새 계산 추가 아님).
    #   lead_f = N - alarm_idx; lead_sec = lead_f * FEMTO_INTERVAL_SEC; lead_h = lead_sec / 3600
    interval_sec = FROZEN_NS.get("FEMTO_INTERVAL_SEC", DECISION_INTERVAL_SEC)
    lead_hours = ((r["N"] - r["alarm_idx"]) * interval_sec / 3600) if detected else None
    return {
        "detected": bool(detected),
        "onset_index": r["alarm_idx"],
        "n_reg": r["n_reg"],
        "threshold": r["thr"],
        "scores": r["dists"],
        "reg_far": r["reg_far"],
        "lead_hours": lead_hours,
    }

ADAPTER_SHA256 = hashlib.sha256((inspect.getsource(m0_features) +
                                  inspect.getsource(m0_detect)).encode()).hexdigest()

def feat_vec(out):
    if isinstance(out, dict):
        names, vals = list(out.keys()), list(out.values())
    elif isinstance(out, pd.Series):
        names, vals = list(out.index), out.values
    else:
        vals = np.asarray(out, dtype=float).ravel(); names = [f"f{i}" for i in range(len(vals))]
    v = np.asarray(vals, dtype=float)
    if v.shape != (9,):
        raise ValueError(f"FEATURE_COUNT_NOT_9: {v.shape}")
    return [str(n) for n in names], v

# 프로브: 두 입력 길이 모두에서 9개 유한값, fs 무시 여부(정보성)
_rng = np.random.default_rng(0)
nA, vA = feat_vec(m0_features(_rng.standard_normal(SOURCE_SAMPLES), SOURCE_FS))
nB, vB = feat_vec(m0_features(_rng.standard_normal(OUTPUT_SAMPLES), OUTPUT_FS))
_x = _rng.standard_normal(OUTPUT_SAMPLES)
FS_IGNORED = bool(np.array_equal(feat_vec(m0_features(_x, SOURCE_FS))[1],
                                  feat_vec(m0_features(_x, OUTPUT_FS))[1]))
if nA != nB or not (np.isfinite(vA).all() and np.isfinite(vB).all()):
    stop("COMPAT_BLOCKED_ADAPTER_NOT_SET", "프로브 실패(특징 이름 불일치 또는 비유한값)")
FEATURE_NAMES = nA
FN_KEY = hashlib.sha256((FROZEN_HASH_BEFORE[str(FROZEN_NB)] + ADAPTER_SHA256 +
                          FROZEN_FN_SOURCE_SHA[FROZEN_FEATURE_FN]).encode()).hexdigest()[:16]
print("특징:", FEATURE_NAMES)
print("fs 값 무시 여부(정보성):", FS_IGNORED, "← True면 주파수 기반 특징이 fs를 안 쓰는지 확인 필요")
print("[Cell 2B] 어댑터 프로브 통과 ✅ FN_KEY =", FN_KEY)

특징: ['RMS', 'Peak', 'Crest', 'Kurtosis', 'Skew', 'P2P', 'Std', 'EnvRMS', 'HF_ratio']
fs 값 무시 여부(정보성): False ← True면 주파수 기반 특징이 fs를 안 쓰는지 확인 필요
[Cell 2B] 어댑터 프로브 통과 ✅ FN_KEY = 47c36a35f318497d


In [ ]:
# ================================================================
# Cell 3 — 원본(2560@25.6kHz) 쪽: 동결 함수로 특징 계산 + 재현 게이트 (r3)
# [r3 수정] 정답표 값은 반올림 저장(lead_hours 소수1자리, reg_FAR 소수4자리) →
#           허용오차 = 정답표 원문 자릿수의 0.5단위(자동). N(파일 수) 일치 검사 추가.
# 특징 계산·캐시 로직은 r1/r2와 동일 (캐시 키 FN_KEY 그대로 → 캐시 재사용)
# ================================================================
REF_COLS = {"bearing": "bearing", "detected": "status", "onset_index": "alarm_idx",
            "reg_far": "reg_FAR", "lead_hours": "lead_hours", "n": "N"}

_need = ["row_index", "source_absolute_path", "source_sha256_at_read", "logical_bearing_id"]
_miss = [c for c in _need if c not in learn_df.columns]
if _miss:
    stop("COMPAT_BLOCKED_GATE", f"dataset_manifest 필수 열 없음: {_miss}", available=list(learn_df.columns))

def load_h_from_bytes(data, path):  # D-022 규칙 (FULL_CONVERT v1_2와 동일)
    first = data.split(b"\n", 1)[0].split(b"\r", 1)[0].decode("utf-8", errors="strict")
    c = [d for d in [",", ";"] if first.count(d) == COL_COUNT_REQUIRED - 1]
    if len(c) != 1:
        raise ValueError(f"D022_SCHEMA_ERROR: {path}")
    fr = pd.read_csv(io.BytesIO(data), header=None, sep=c[0], decimal=".", low_memory=False)
    if fr.shape != (SOURCE_SAMPLES, COL_COUNT_REQUIRED):
        raise ValueError(f"SOURCE_SCHEMA_INVALID: {path} {fr.shape}")
    x = pd.to_numeric(fr.iloc[:, COL_H], errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(x).all():
        raise ValueError(f"SOURCE_NAN_INF: {path}")
    return x

A = {}
t0 = time.time()
for b in LEARNING_BEARINGS:
    rows = learn_df[learn_df["logical_bearing_id"] == b].sort_values("row_index").reset_index(drop=True)
    cache = CACHE_DIR / f"A_{b}_{FN_KEY}.npz"
    if cache.is_file():
        z = np.load(cache, allow_pickle=False)
        if int(z["n"]) == len(rows):
            A[b] = {"F": z["F"], "rms": z["rms"], "paths": rows["source_absolute_path"].tolist()}
            print(f"[A] CACHE {b}: {len(rows)}"); continue
    F = np.zeros((len(rows), 9)); rms = np.zeros(len(rows)); sha_bad = []
    print(f"[A] {b}: {len(rows):,} files")
    for i, r in rows.iterrows():
        data = with_io_retry(lambda p: Path(p).read_bytes(), r["source_absolute_path"])
        if hashlib.sha256(data).hexdigest() != str(r["source_sha256_at_read"]).strip().lower():
            sha_bad.append(r["source_absolute_path"])
        x = load_h_from_bytes(data, r["source_absolute_path"])
        F[i] = feat_vec(m0_features(x, SOURCE_FS))[1]
        rms[i] = float(np.sqrt(np.mean(x ** 2)))
        if (i + 1) % 500 == 0:
            print(f"  ... {i+1:,}/{len(rows):,} (누적 {(time.time()-t0)/60:.1f}분)")
    if sha_bad:
        stop("COMPAT_BLOCKED_SOURCE_CHANGED", f"{b}: 원본 SHA 불일치 {len(sha_bad)}건", files=sha_bad[:20])
    np.savez(cache, F=F, rms=rms, n=len(rows))
    A[b] = {"F": F, "rms": rms, "paths": rows["source_absolute_path"].tolist()}

for b in LEARNING_BEARINGS:
    A[b]["det"] = m0_detect(A[b]["F"])

# ── 재현 게이트 ──
ref = pd.read_csv(FROZEN_RESULT_CSV)
ref_str = pd.read_csv(FROZEN_RESULT_CSV, dtype=str, keep_default_na=False)  # 원문 자릿수 판단용
_miss_ref = [v for v in REF_COLS.values() if v and v not in ref.columns]
if _miss_ref:
    print("결과 CSV 열:", list(ref.columns)); display(ref.head(10))
    stop("COMPAT_BLOCKED_ADAPTER_NOT_SET", f"REF_COLS 열 없음: {_miss_ref}", ref_columns=list(ref.columns))

def _bid(v):
    m = re.search(r"Bearing\d_\d", str(v)); return m.group(0) if m else str(v)

def _status_detected(v):
    s = str(v).strip().lower()
    if s in ("", "nan", "none"): return None
    if "미탐지" in s or "❌" in s or s in ("false", "0", "no", "n", "not_detected"): return False
    if "탐지" in s or "✅" in s or s in ("true", "1", "yes", "y", "detected"): return True
    return None

def _tol_from_text(s):
    # 정답표 원문 자릿수의 0.5단위 (예: "3.7" → 0.05, "0.0033" → 0.00005) + 부동소수 여유
    s = str(s).strip().lower()
    if "e" in s: return 1e-9
    d = len(s.split(".")[1]) if "." in s else 0
    return 0.5 * 10 ** (-d) + 1e-12

ref["_b"] = ref[REF_COLS["bearing"]].map(_bid)
ref_str["_b"] = ref_str[REF_COLS["bearing"]].map(_bid)
repro, ref_inconsistent = [], []
for b in LEARNING_BEARINGS:
    rr = ref[ref["_b"] == b]; rs = ref_str[ref_str["_b"] == b]
    if len(rr) != 1:
        stop("COMPAT_BLOCKED_REFERENCE_NOT_REPRODUCED", f"결과 CSV에 {b} 행 {len(rr)}개")
    rr, rs = rr.iloc[0], rs.iloc[0]; d = A[b]["det"]

    _on = pd.to_numeric(pd.Series([rr[REF_COLS["onset_index"]]]), errors="coerce").iloc[0]
    ref_on = None if pd.isna(_on) else int(_on)
    st = _status_detected(rr[REF_COLS["detected"]])
    if st is None or st != (ref_on is not None):
        ref_inconsistent.append({"bearing": b, "status": rr[REF_COLS["detected"]],
                                 "alarm_idx": rr[REF_COLS["onset_index"]]})
        continue

    a_n = int(A[b]["F"].shape[0])
    row = {"bearing": b, "ref_N": int(rr[REF_COLS["n"]]), "a_N": a_n,
           "ref_detected": st, "a_detected": bool(d["detected"]),
           "ref_onset": ref_on, "a_onset": d["onset_index"]}
    ok = (row["ref_N"] == a_n) and (st == row["a_detected"]) and (ref_on == d["onset_index"])

    far_tol = _tol_from_text(rs[REF_COLS["reg_far"]])
    row.update({"ref_far": float(rr[REF_COLS["reg_far"]]), "a_far": float(d["reg_far"]), "far_tol": far_tol})
    ok &= abs(row["ref_far"] - row["a_far"]) <= far_tol

    if st and d.get("lead_hours") is not None:
        lead_tol = _tol_from_text(rs[REF_COLS["lead_hours"]])
        row.update({"ref_lead_h": float(rr[REF_COLS["lead_hours"]]),
                    "a_lead_h": float(d["lead_hours"]), "lead_tol": lead_tol})
        ok &= abs(row["ref_lead_h"] - row["a_lead_h"]) <= lead_tol
    row["match"] = bool(ok); repro.append(row)

if ref_inconsistent:
    display(pd.DataFrame(ref_inconsistent))
    stop("COMPAT_BLOCKED_REFERENCE_UNREADABLE",
         "정답표 status와 alarm_idx 유무가 불일치하거나 status 해석 불가", rows=ref_inconsistent)

repro_df = pd.DataFrame(repro)
repro_df.to_csv(REPORT_DIR / "reference_reproduction_check.csv", index=False, encoding="utf-8-sig")
display(repro_df)
REFERENCE_REPRODUCED = bool(repro_df["match"].all())
print(f"검출 {int(repro_df['a_detected'].sum())}/6 | "
      f"정밀값 평균 리드타임 {repro_df['a_lead_h'].mean():.4f}h, 평균 reg_FAR {repro_df['a_far'].mean()*100:.3f}%")
print("※ 정답표는 반올림 저장값(lead 소수1자리, FAR 소수4자리). 허용오차 = 원문 자릿수 0.5단위")
if not REFERENCE_REPRODUCED:
    stop("COMPAT_BLOCKED_REFERENCE_NOT_REPRODUCED", "동결 결과 재현 실패 — 2048 비교 진행 안 함",
         mismatch=repro_df[~repro_df["match"]].to_dict("records"))
print("[Cell 3] 동결 결과 재현 ✅")


[A] CACHE Bearing1_1: 2803
[A] CACHE Bearing1_2: 871
[A] CACHE Bearing2_1: 911
[A] CACHE Bearing2_2: 797
[A] CACHE Bearing3_1: 515
[A] CACHE Bearing3_2: 1637


,bearing,ref_N,a_N,ref_detected,a_detected,ref_onset,a_onset,ref_far,a_far,far_tol,ref_lead_h,a_lead_h,lead_tol,match
0,Bearing1_1,2803,2803,True,True,1462,1462,0.0033,0.003333,0.00005,3.7,3.725000,0.05,True
1,Bearing1_2,871,871,True,True,826,826,0.0000,0.000000,0.05000,0.1,0.125000,0.05,True
2,Bearing2_1,911,911,True,True,849,849,0.0088,0.008811,0.00005,0.2,0.172222,0.05,True
3,Bearing2_2,797,797,True,True,274,274,0.0050,0.005025,0.00050,1.5,1.452778,0.05,True
4,Bearing3_1,515,515,True,True,496,496,0.0078,0.007812,0.00005,0.1,0.052778,0.05,True
5,Bearing3_2,1637,1637,True,True,1449,1449,0.0024,0.002445,0.00005,0.5,0.522222,0.05,True


검출 6/6 | 정밀값 평균 리드타임 1.0083h, 평균 reg_FAR 0.457%
※ 정답표는 반올림 저장값(lead 소수1자리, FAR 소수4자리). 허용오차 = 원문 자릿수 0.5단위
[Cell 3] 동결 결과 재현 ✅


In [ ]:
# ================================================================
# Cell 4 — 2048(20.48kHz) 쪽 (r2)
# [r2] index.csv 필수 열 확인, 경로 strip 후 비교, 첫 불일치 위치 보고,
#      dtype/NaN 검사 추가. 특징·판정 계산은 r1과 동일.
# ================================================================
_need_idx = ["row_index", "source_absolute_path"]
B = {}
for b in LEARNING_BEARINGS:
    rec = DATASET_ROOT / "LEARNING" / b
    arr = with_io_retry(np.load, rec / "signals.npy")
    idx = pd.read_csv(rec / "index.csv", low_memory=False)
    miss = [c for c in _need_idx if c not in idx.columns]
    if miss:
        print(f"{b} index.csv 열:", list(idx.columns))
        stop("COMPAT_BLOCKED_GATE", f"{b}: index.csv 필수 열 없음 {miss}",
             available=list(idx.columns))
    idx = idx.sort_values("row_index").reset_index(drop=True)
    paths_a = [str(p).strip() for p in A[b]["paths"]]
    paths_b = idx["source_absolute_path"].astype(str).str.strip().tolist()

    if arr.ndim != 2 or arr.shape != (len(paths_a), OUTPUT_SAMPLES):
        stop("COMPAT_BLOCKED_GATE", f"{b}: 배열 shape {arr.shape} ≠ ({len(paths_a)}, {OUTPUT_SAMPLES})")
    if arr.dtype != np.float32:
        stop("COMPAT_BLOCKED_GATE", f"{b}: 저장 dtype {arr.dtype} (계약은 float32)")
    if not np.isfinite(arr).all():
        stop("COMPAT_BLOCKED_GATE", f"{b}: 2048 배열에 NaN/Inf")
    if paths_b != paths_a:
        k = (next(i for i, (x, y) in enumerate(zip(paths_a, paths_b)) if x != y)
             if len(paths_a) == len(paths_b) else None)
        stop("COMPAT_BLOCKED_GATE", f"{b}: 원본-2048 순서 불일치",
             n_a=len(paths_a), n_b=len(paths_b), first_mismatch=k,
             a=paths_a[k] if k is not None else None, b=paths_b[k] if k is not None else None)

    F = np.zeros((arr.shape[0], 9)); rms = np.zeros(arr.shape[0])
    for i in range(arr.shape[0]):
        x = arr[i].astype(np.float64)  # 저장 float32 → 계산 float64
        F[i] = feat_vec(m0_features(x, OUTPUT_FS))[1]
        rms[i] = float(np.sqrt(np.mean(x ** 2)))
    B[b] = {"F": F, "rms": rms, "det": m0_detect(F)}
    d = B[b]["det"]
    print(f"[B] {b}: {arr.shape[0]:,} 완료 | 검출={d['detected']} onset={d['onset_index']} "
          f"reg_FAR={d['reg_far']:.4f}")
print("[Cell 4] 완료 ✅")


[B] Bearing1_1: 2,803 완료 | 검출=True onset=1462 reg_FAR=0.0033
[B] Bearing1_2: 871 완료 | 검출=True onset=826 reg_FAR=0.0000
[B] Bearing2_1: 911 완료 | 검출=True onset=849 reg_FAR=0.0088
[B] Bearing2_2: 797 완료 | 검출=True onset=705 reg_FAR=0.0050
[B] Bearing3_1: 515 완료 | 검출=True onset=496 reg_FAR=0.0078
[B] Bearing3_2: 1,637 완료 | 검출=True onset=1584 reg_FAR=0.0024
[Cell 4] 완료 ✅


In [ ]:
# ================================================================
# Cell 5 — 비교 + 판정 (합의 기준, 변경 금지)
# ================================================================
sp_rows, det_rows, rms_rows = [], [], []
for b in LEARNING_BEARINGS:
    for j, name in enumerate(FEATURE_NAMES):
        rho = spearmanr(A[b]["F"][:, j], B[b]["F"][:, j]).correlation
        med_a = np.median(A[b]["F"][:, j])
        sp_rows.append({"bearing": b, "feature": name,
                         "spearman": None if np.isnan(rho) else float(rho),
                         "median_ratio_B_over_A": float(np.median(B[b]["F"][:, j]) / med_a) if med_a != 0 else None})
    da, db = A[b]["det"], B[b]["det"]
    s_rho = spearmanr(np.asarray(da["scores"]), np.asarray(db["scores"])).correlation
    diff = (db["onset_index"] - da["onset_index"]) if (da["detected"] and db["detected"]) else None
    det_rows.append({"bearing": b, "a_detected": bool(da["detected"]), "b_detected": bool(db["detected"]),
                      "a_onset": da["onset_index"], "b_onset": db["onset_index"], "onset_diff_windows": diff,
                      "a_lead_h": da.get("lead_hours"), "b_lead_h": db.get("lead_hours"),
                      "a_reg_far": da["reg_far"], "b_reg_far": db["reg_far"],
                      "a_threshold": da["threshold"], "b_threshold": db["threshold"],
                      "n_reg": da["n_reg"], "score_spearman_info": None if np.isnan(s_rho) else float(s_rho)})
    ratio = B[b]["rms"] / np.where(A[b]["rms"] == 0, np.nan, A[b]["rms"])
    rms_rows.append({"bearing": b, "signal_rms_ratio_median": float(np.nanmedian(ratio)),
                      "p05": float(np.nanpercentile(ratio, 5)), "p95": float(np.nanpercentile(ratio, 95))})

sp_df, det_df, rms_df = pd.DataFrame(sp_rows), pd.DataFrame(det_rows), pd.DataFrame(rms_rows)
sp_df.to_csv(REPORT_DIR / "feature_spearman_by_bearing.csv", index=False, encoding="utf-8-sig")
det_df.to_csv(REPORT_DIR / "detection_comparison.csv", index=False, encoding="utf-8-sig")
rms_df.to_csv(REPORT_DIR / "signal_rms_ratio_by_bearing.csv", index=False, encoding="utf-8-sig")

feature_ok = bool(sp_df["spearman"].notna().all() and (sp_df["spearman"] >= CRITERIA["feature_spearman_min"]).all())
detect_set_ok = bool((det_df["a_detected"] == det_df["b_detected"]).all())
both = det_df[det_df["a_detected"] & det_df["b_detected"]]
onset_ok = bool((both["onset_diff_windows"].abs() <= CRITERIA["onset_diff_max_windows"]).all())
COMPAT_PASS = feature_ok and detect_set_ok and onset_ok

print("특징별 최소 Spearman (6베어링 중):")
display(sp_df.groupby("feature")["spearman"].min().sort_values())
display(det_df); display(rms_df)
print(f"feature_ok={feature_ok} detect_set_ok={detect_set_ok} onset_ok={onset_ok}")
print("STATUS:", "COMPAT_PASS" if COMPAT_PASS else "COMPAT_FAIL_MEASURED (기준 변경 없이 대표 보고)")

특징별 최소 Spearman (6베어링 중):


,spearman
feature,
Std,-0.005044
RMS,-0.002786
HF_ratio,0.036523
EnvRMS,0.146862
P2P,0.331287
Peak,0.345306
Skew,0.456746
Crest,0.479201
Kurtosis,0.603757


,bearing,a_detected,b_detected,a_onset,b_onset,onset_diff_windows,a_lead_h,b_lead_h,a_reg_far,b_reg_far,a_threshold,b_threshold,n_reg,score_spearman_info
0,Bearing1_1,True,True,1462,1462,0,3.725000,3.725000,0.003333,0.003333,13.110604,12.185807,600,0.992532
1,Bearing1_2,True,True,826,826,0,0.125000,0.125000,0.000000,0.000000,12.249213,12.034208,217,0.861076
2,Bearing2_1,True,True,849,849,0,0.172222,0.172222,0.008811,0.008811,11.345420,11.654767,227,0.414421
3,Bearing2_2,True,True,274,705,431,1.452778,0.255556,0.005025,0.005025,10.130620,9.948625,199,0.721502
4,Bearing3_1,True,True,496,496,0,0.052778,0.052778,0.007812,0.007812,10.359342,10.360141,128,0.932177
5,Bearing3_2,True,True,1449,1584,135,0.522222,0.147222,0.002445,0.002445,9.279422,9.242886,409,0.943493


,bearing,signal_rms_ratio_median,p05,p95
0,Bearing1_1,0.966654,0.936698,0.991521
1,Bearing1_2,0.864827,0.693742,0.980468
2,Bearing2_1,0.494621,0.427270,0.992025
3,Bearing2_2,0.543536,0.386359,0.986663
4,Bearing3_1,0.979711,0.959277,0.997344
5,Bearing3_2,0.975702,0.819941,0.990461


feature_ok=False detect_set_ok=True onset_ok=False
STATUS: COMPAT_FAIL_MEASURED (기준 변경 없이 대표 보고)


In [ ]:
# ================================================================
# Cell 6 — 최종 요약 + 동결 파일 실제 재해시(전후 비교)
# ================================================================
FROZEN_HASH_AFTER = {p: sha256_file(p) for p in FROZEN_HASH_BEFORE}
frozen_unchanged = FROZEN_HASH_AFTER == FROZEN_HASH_BEFORE
status = ("COMPAT_INVALID_FROZEN_MODIFIED" if not frozen_unchanged
          else "COMPAT_PASS" if COMPAT_PASS else "COMPAT_FAIL_MEASURED")
summary = {
    "version": VERSION, "status": status,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "criteria": CRITERIA, "criteria_sha256": CRITERIA_SHA256,
    "checks": {"reference_reproduced": REFERENCE_REPRODUCED, "feature_spearman_ok": feature_ok,
               "detected_set_match": detect_set_ok, "onset_diff_ok": onset_ok,
               "frozen_files_unchanged": frozen_unchanged},
    "feature_names": FEATURE_NAMES,
    "min_spearman_by_feature": sp_df.groupby("feature")["spearman"].min().to_dict(),
    "detection": det_df.to_dict("records"),
    "signal_rms_ratio": rms_df.to_dict("records"),
    "fs_ignored_probe": FS_IGNORED,
    "frozen_notebook": str(FROZEN_NB), "frozen_fn": [FROZEN_FEATURE_FN, FROZEN_DETECT_FN],
    "frozen_fn_source_sha256": FROZEN_FN_SOURCE_SHA, "adapter_sha256": ADAPTER_SHA256,
    "frozen_hash_before": FROZEN_HASH_BEFORE, "frozen_hash_after": FROZEN_HASH_AFTER,
    "full_convert_dir": str(FC_DIR), "dataset_contract_sha256": ds_contract.get("contract_sha256"),
    "data_scope": "LEARNING 6 bearings only (TEST/FULL_TEST 미사용)",
    "decision_interval_sec": DECISION_INTERVAL_SEC,
    "not_included": ["운전조건 버킷 시뮬(C1/C2/C3) — 별도 단계", "2048 운용 계약/Shadow 모델"],
    "interpretation_limit": "FEMTO 2048@20.48kHz 결과이며 현장 2048@26.667kHz 성능으로 해석 금지",
    "threshold_or_params_modified": False, "training_performed": False, "m0_reimplemented": False,
}
write_json(REPORT_DIR / "COMPAT_FINAL_SUMMARY.json", summary)
print("=" * 60); print("STATUS:", status)
for k, v in summary["checks"].items():
    print(f"  {'✅' if v else '❌'} {k}")
print("리포트:", REPORT_DIR)

STATUS: COMPAT_FAIL_MEASURED
  ✅ reference_reproduced
  ❌ feature_spearman_ok
  ✅ detected_set_match
  ❌ onset_diff_ok
  ✅ frozen_files_unchanged
리포트: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20261029_110751_m0_2048_compat_v1


In [ ]:
# ================================================================
# Cell 7 — 결과 텍스트 출력 (읽기 전용, 저장된 리포트만 읽음)
# ================================================================
import json, pandas as pd
from pathlib import Path
BR = Path("/content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs")
runs = sorted(BR.glob("*_m0_2048_compat_v1"), key=lambda p: p.name)
R = runs[-1]
print("리포트:", R.name, "| 전체 실행 폴더 수:", len(runs))
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)

s = json.loads((R / "COMPAT_FINAL_SUMMARY.json").read_text(encoding="utf-8"))
print("\nSTATUS:", s.get("status"), "|", s.get("reason", ""))
print("checks:", json.dumps(s.get("checks", {}), ensure_ascii=False))
print("fs_ignored_probe:", s.get("fs_ignored_probe"))

for f in ["reference_reproduction_check.csv", "detection_comparison.csv",
          "signal_rms_ratio_by_bearing.csv"]:
    p = R / f
    print(f"\n=== {f} ===")
    print(pd.read_csv(p).round(4).to_string(index=False) if p.is_file() else "(없음)")

p = R / "feature_spearman_by_bearing.csv"
if p.is_file():
    sp = pd.read_csv(p)
    print("\n=== Spearman: 특징 × 베어링 ===")
    print(sp.pivot(index="feature", columns="bearing", values="spearman").round(4).to_string())
    print("\n=== median_ratio (2048/원본) ===")
    print(sp.pivot(index="feature", columns="bearing", values="median_ratio_B_over_A").round(3).to_string())
    print(f"\n0.95 미만: {int((sp['spearman'] < 0.95).sum())}/54")


리포트: 20261029_110751_m0_2048_compat_v1 | 전체 실행 폴더 수: 2

STATUS: COMPAT_FAIL_MEASURED | 
checks: {"reference_reproduced": true, "feature_spearman_ok": false, "detected_set_match": true, "onset_diff_ok": false, "frozen_files_unchanged": true}
fs_ignored_probe: False

=== reference_reproduction_check.csv ===
   bearing  ref_N  a_N  ref_detected  a_detected  ref_onset  a_onset  ref_far  a_far  far_tol  ref_lead_h  a_lead_h  lead_tol  match
Bearing1_1   2803 2803          True        True       1462     1462   0.0033 0.0033   0.0001         3.7    3.7250      0.05   True
Bearing1_2    871  871          True        True        826      826   0.0000 0.0000   0.0500         0.1    0.1250      0.05   True
Bearing2_1    911  911          True        True        849      849   0.0088 0.0088   0.0001         0.2    0.1722      0.05   True
Bearing2_2    797  797          True        True        274      274   0.0050 0.0050   0.0005         1.5    1.4528      0.05   True
Bearing3_1    515  515      

In [ ]:
# ================================================================
# Cell 8 — 대역폭 원인 분리 진단 (B안) — 진단 전용, COMPAT 판정 변경 없음
# 원본 2560@25.6kHz 신호에 저역통과만 적용 → 동결 M0 그대로 실행
# fs·샘플 수는 원본 유지 → "대역폭" 효과만 분리
# 쓰기: CACHE_DIR(캐시), 새 진단 폴더만. 원본 CSV·동결 파일·2048 데이터셋 쓰기 없음
# ================================================================
import scipy.signal as sps

_req = ["A", "B", "learn_df", "LEARNING_BEARINGS", "m0_features", "m0_detect", "feat_vec",
        "load_h_from_bytes", "with_io_retry", "CACHE_DIR", "BRIDGE_ROOT", "FN_KEY",
        "FROZEN_HASH_BEFORE", "sha256_file", "write_json", "SOURCE_FS", "SOURCE_SAMPLES", "FEATURE_NAMES"]
_miss = [n for n in _req if n not in globals()]
if _miss:
    raise RuntimeError(f"선행 셀 변수 없음: {_miss} → Cell 1, 2, 2B, 3, 4 순서로 먼저 실행 (3·4는 캐시로 수 분)")

DIAG_VERSION = "M0_2048_BANDWIDTH_DIAG_v1"
DIAG_DIR = BRIDGE_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S_m0_2048_bandwidth_diag_v1")
DIAG_DIR.mkdir(parents=True, exist_ok=False)

def diag_stop(reason, **extra):
    write_json(DIAG_DIR / "BANDWIDTH_DIAG_SUMMARY.json",
               {"version": DIAG_VERSION, "status": "DIAG_BLOCKED", "reason": reason, **extra})
    raise RuntimeError(f"DIAG_BLOCKED: {reason}")

# ── 필터 설계: 대칭(선형위상) Kaiser FIR, 반사 패딩 + valid 합성곱 → 지연 0, 길이 유지 ──
CUTOFFS = {"LP_NONE": None, "LP10240": 10240.0, "LP8000": 8000.0, "LP6000": 6000.0, "LP4000": 4000.0}
FIR_RIPPLE_DB, FIR_WIDTH_HZ = 60.0, 400.0          # 저지대역 -60dB, 전이폭 400Hz
_nt, FIR_BETA = sps.kaiserord(FIR_RIPPLE_DB, FIR_WIDTH_HZ / (SOURCE_FS / 2))
FIR_NUMTAPS = _nt + (1 - _nt % 2)                   # 홀수 강제
PAD = FIR_NUMTAPS // 2
FIR = {c: sps.firwin(FIR_NUMTAPS, fc, window=("kaiser", FIR_BETA), fs=SOURCE_FS)
       for c, fc in CUTOFFS.items() if fc}
FILT_KEY = hashlib.sha256(json.dumps({"cut": CUTOFFS, "ripple": FIR_RIPPLE_DB, "width": FIR_WIDTH_HZ,
                                      "numtaps": int(FIR_NUMTAPS), "beta": float(FIR_BETA)},
                                     sort_keys=True).encode()).hexdigest()[:12]
print(f"FIR: taps={FIR_NUMTAPS}, beta={FIR_BETA:.3f}, 컷오프=-6dB 지점, FILT_KEY={FILT_KEY}")

def lowpass(x, cond):
    if CUTOFFS[cond] is None:
        return x
    y = np.convolve(np.pad(x, PAD, mode="reflect"), FIR[cond], mode="valid")
    assert y.shape == x.shape
    return y

# ── 원본 H축 원시신호 캐시 (최초 1회만 CSV 읽기, 이후 수 분) ──
def load_raw(b):
    rows = learn_df[learn_df["logical_bearing_id"] == b].sort_values("row_index").reset_index(drop=True)
    paths = rows["source_absolute_path"].astype(str).str.strip().tolist()
    if paths != [str(p).strip() for p in A[b]["paths"]]:
        diag_stop(f"{b}: Cell 3 경로 순서와 불일치")
    pkey = hashlib.sha256("\n".join(paths).encode()).hexdigest()[:12]
    cp = CACHE_DIR / f"RAW_H_{b}_{pkey}.npy"
    if cp.is_file():
        X = np.load(cp)
        if X.shape == (len(rows), SOURCE_SAMPLES):
            print(f"[RAW] CACHE {b}: {len(rows):,}")
            return X
    X = np.zeros((len(rows), SOURCE_SAMPLES)); bad = []; t = time.time()
    print(f"[RAW] {b}: {len(rows):,} files 읽는 중")
    for i, r in rows.iterrows():
        data = with_io_retry(lambda p: Path(p).read_bytes(), r["source_absolute_path"])
        if hashlib.sha256(data).hexdigest() != str(r["source_sha256_at_read"]).strip().lower():
            bad.append(r["source_absolute_path"])
        X[i] = load_h_from_bytes(data, r["source_absolute_path"])
        if (i + 1) % 500 == 0:
            print(f"  ... {i+1:,}/{len(rows):,} ({(time.time()-t)/60:.1f}분)")
    if bad:
        diag_stop(f"{b}: 원본 SHA 불일치 {len(bad)}건", files=bad[:20])
    tmp = cp.with_name(cp.stem + "_tmp.npy"); np.save(tmp, X); os.replace(tmp, cp)
    return X

BANDS = [(0, 2000), (2000, 4000), (4000, 6000), (6000, 8000), (8000, 10240), (10240, 12800)]
freqs = np.fft.rfftfreq(SOURCE_SAMPLES, 1 / SOURCE_FS)

det_rows, sp_rows, band_rows = [], [], []
t0 = time.time()
for b in LEARNING_BEARINGS:
    X = load_raw(b)
    da, db = A[b]["det"], B[b]["det"]

    # 대역별 에너지 비율 (정상구간 = 등록구간 n_reg, 말기 = 마지막 10%)
    P = np.abs(np.fft.rfft(X - X.mean(axis=1, keepdims=True), axis=1)) ** 2
    tot = P.sum(axis=1)
    n_late = max(1, len(X) // 10)
    for lo, hi in BANDS:
        m = (freqs >= lo) & (freqs < hi) if hi < 12800 else (freqs >= lo)
        frac = P[:, m].sum(axis=1) / tot
        band_rows.append({"bearing": b, "band": f"{lo/1000:g}-{hi/1000:g}k",
                          "healthy_median": float(np.median(frac[:da["n_reg"]])),
                          "late_median": float(np.median(frac[-n_late:]))})
    del P

    for cond in CUTOFFS:
        fp = CACHE_DIR / f"D_{b}_{cond}_{FN_KEY}_{FILT_KEY}.npy"
        if fp.is_file() and np.load(fp).shape == (len(X), 9):
            F = np.load(fp)
        else:
            F = np.zeros((len(X), 9))
            for i in range(len(X)):
                F[i] = feat_vec(m0_features(lowpass(X[i], cond), SOURCE_FS))[1]
            tmp = fp.with_name(fp.stem + "_tmp.npy"); np.save(tmp, F); os.replace(tmp, fp)

        if cond == "LP_NONE" and not np.allclose(F, A[b]["F"], rtol=1e-12, atol=1e-12):
            diag_stop(f"{b}: 필터 없음 조건이 Cell 3 특징과 불일치 — 파이프라인 점검 필요")

        d = m0_detect(F)
        det_rows.append({"bearing": b, "cond": cond, "detected": d["detected"], "onset": d["onset_index"],
                         "onset_diff_vs_A": (d["onset_index"] - da["onset_index"]) if d["detected"] else None,
                         "onset_diff_vs_B": (d["onset_index"] - db["onset_index"]) if d["detected"] else None,
                         "lead_h": d["lead_hours"], "reg_far": d["reg_far"],
                         "threshold": d["threshold"], "n_reg": d["n_reg"]})
        for j, name in enumerate(FEATURE_NAMES):
            rA = spearmanr(A[b]["F"][:, j], F[:, j]).correlation
            rB = spearmanr(B[b]["F"][:, j], F[:, j]).correlation if cond == "LP10240" else np.nan
            sp_rows.append({"bearing": b, "cond": cond, "feature": name,
                            "spearman_vs_A": None if np.isnan(rA) else float(rA),
                            "spearman_vs_B": None if np.isnan(rB) else float(rB)})
    print(f"[DIAG] {b} 완료 (누적 {(time.time()-t0)/60:.1f}분)")
    del X

det_df8, sp_df8, band_df8 = pd.DataFrame(det_rows), pd.DataFrame(sp_rows), pd.DataFrame(band_rows)
det_df8.to_csv(DIAG_DIR / "bandwidth_diag_detection.csv", index=False, encoding="utf-8-sig")
sp_df8.to_csv(DIAG_DIR / "bandwidth_diag_spearman.csv", index=False, encoding="utf-8-sig")
band_df8.to_csv(DIAG_DIR / "band_energy_fraction.csv", index=False, encoding="utf-8-sig")

# ── 결과 출력 (텍스트로 복사 가능) ──
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
ref_on = {b: A[b]["det"]["onset_index"] for b in LEARNING_BEARINGS}
b2048_on = {b: B[b]["det"]["onset_index"] for b in LEARNING_BEARINGS}
on_tab = det_df8.pivot(index="cond", columns="bearing", values="onset").reindex(list(CUTOFFS))
on_tab.loc["A_원본"] = pd.Series(ref_on); on_tab.loc["B_2048"] = pd.Series(b2048_on)
print("\n=== onset (윈도 번호) ===");        print(on_tab.to_string())
print("\n=== lead_h ===")
print(det_df8.pivot(index="cond", columns="bearing", values="lead_h").reindex(list(CUTOFFS)).round(3).to_string())
print("\n=== reg_FAR ===")
print(det_df8.pivot(index="cond", columns="bearing", values="reg_far").reindex(list(CUTOFFS)).round(4).to_string())
print("\n=== 조건별 특징 최소 Spearman (vs 원본 A, 6베어링 중 최소) ===")
print(sp_df8.groupby(["feature", "cond"])["spearman_vs_A"].min().unstack()[list(CUTOFFS)].round(3).to_string())
print("\n=== LP10240 vs 2048(B) 특징 Spearman (높으면 '대역폭만으로 설명됨') ===")
print(sp_df8[sp_df8["cond"] == "LP10240"].pivot(index="feature", columns="bearing",
      values="spearman_vs_B").round(3).to_string())
print("\n=== 대역별 에너지 비율 (정상구간 / 말기) ===")
print(band_df8.pivot(index="band", columns="bearing", values="healthy_median").round(3).to_string())
print(band_df8.pivot(index="band", columns="bearing", values="late_median").round(3).to_string())

lp = det_df8[det_df8["cond"] == "LP10240"].set_index("bearing")
lp10240_matches_B = {b: bool(lp.loc[b, "onset"] == b2048_on[b]) for b in LEARNING_BEARINGS}
print("\nLP10240 onset == 2048 onset:", lp10240_matches_B)

FROZEN_HASH_AFTER8 = {p: sha256_file(p) for p in FROZEN_HASH_BEFORE}
frozen_ok8 = FROZEN_HASH_AFTER8 == FROZEN_HASH_BEFORE
write_json(DIAG_DIR / "BANDWIDTH_DIAG_SUMMARY.json", {
    "version": DIAG_VERSION, "status": "DIAG_COMPLETE",
    "note": "진단 전용 — COMPAT 판정(COMPAT_FAIL_MEASURED)·기준·동결 파라미터 변경 없음",
    "compat_report": "bridge_outputs/20261029_110751_m0_2048_compat_v1",
    "filter": {"type": "kaiser FIR, zero-phase(symmetric, reflect pad, valid conv)", "numtaps": int(FIR_NUMTAPS),
               "beta": float(FIR_BETA), "ripple_db": FIR_RIPPLE_DB, "width_hz": FIR_WIDTH_HZ,
               "cutoff_definition": "-6dB", "cutoffs": CUTOFFS, "filt_key": FILT_KEY},
    "fs_and_samples_kept": {"fs": SOURCE_FS, "samples": SOURCE_SAMPLES},
    "lp_none_reproduced_cell3": True, "lp10240_onset_matches_2048": lp10240_matches_B,
    "detection": det_df8.to_dict("records"), "frozen_files_unchanged": frozen_ok8,
    "threshold_or_params_modified": False, "training_performed": False, "m0_reimplemented": False,
    "data_scope": "LEARNING 6 bearings only",
})
print("\n" + "=" * 60)
print("STATUS: DIAG_COMPLETE | 동결 파일 불변:", "✅" if frozen_ok8 else "❌")
print("진단 폴더:", DIAG_DIR)


FIR: taps=235, beta=5.653, 컷오프=-6dB 지점, FILT_KEY=651fd4063ef8
[RAW] Bearing1_1: 2,803 files 읽는 중
  ... 500/2,803 (0.1분)
  ... 1,000/2,803 (0.1분)
  ... 1,500/2,803 (0.2분)
  ... 2,000/2,803 (0.2분)
  ... 2,500/2,803 (0.3분)
[DIAG] Bearing1_1 완료 (누적 0.8분)
[RAW] Bearing1_2: 871 files 읽는 중
  ... 500/871 (0.0분)
[DIAG] Bearing1_2 완료 (누적 1.1분)
[RAW] Bearing2_1: 911 files 읽는 중
  ... 500/911 (0.0분)
[DIAG] Bearing2_1 완료 (누적 1.4분)
[RAW] Bearing2_2: 797 files 읽는 중
  ... 500/797 (0.0분)
[DIAG] Bearing2_2 완료 (누적 1.6분)
[RAW] Bearing3_1: 515 files 읽는 중
  ... 500/515 (0.0분)
[DIAG] Bearing3_1 완료 (누적 1.8분)
[RAW] Bearing3_2: 1,637 files 읽는 중
  ... 500/1,637 (0.0분)
  ... 1,000/1,637 (0.1분)
  ... 1,500/1,637 (0.1분)
[DIAG] Bearing3_2 완료 (누적 2.2분)

=== onset (윈도 번호) ===
bearing  Bearing1_1  Bearing1_2  Bearing2_1  Bearing2_2  Bearing3_1  Bearing3_2
cond                                                                           
LP_NONE        1462         826         849         274         496        1449
LP10240

In [ ]:
# [읽기 전용] Drive 재연결 + COMPAT 리포트 값 출력 — 파일 쓰기 없음
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import json, glob, os
import pandas as pd

ROOT = "/content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs"
cands = sorted(glob.glob(os.path.join(ROOT, "*m0_2048_compat*")))
print("COMPAT 후보 폴더:", [os.path.basename(c) for c in cands])
if not cands:
    raise SystemExit("compat 폴더 없음 → bridge_outputs 목록: " + str(sorted(os.listdir(ROOT))))
BR = cands[-1]
print("사용 폴더:", BR)
print("파일 목록:", sorted(os.listdir(BR)))

def walk(o, path=""):
    if isinstance(o, dict):
        for k, v in o.items():
            yield from walk(v, f"{path}.{k}" if path else str(k))
    elif isinstance(o, list):
        for i, v in enumerate(o):
            yield from walk(v, f"{path}[{i}]")
    else:
        yield path, o

KEYS = ("far", "hash", "lead", "status", "detect", "onset")

for jp in sorted(glob.glob(os.path.join(BR, "*.json"))):
    print(f"\n=== JSON: {os.path.basename(jp)}")
    with open(jp, encoding="utf-8") as f:
        d = json.load(f)
    for p, v in walk(d):
        if any(s in p.lower() for s in KEYS):
            print(f"  {p} = {v}")

for cp in sorted(glob.glob(os.path.join(BR, "*.csv"))):
    df = pd.read_csv(cp)
    if not any("far" in c.lower() for c in df.columns):
        continue
    cols = [c for c in df.columns
            if any(s in c.lower() for s in ("far", "bearing", "onset", "lead", "source", "variant", "cond", "input"))]
    print(f"\n=== CSV: {os.path.basename(cp)}")
    print(df[cols].to_string(index=False))


Mounted at /content/drive
COMPAT 후보 폴더: ['20261024_024032_m0_2048_compat_v1', '20261029_110751_m0_2048_compat_v1']
사용 폴더: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20261029_110751_m0_2048_compat_v1
파일 목록: ['COMPAT_FINAL_SUMMARY.json', 'detection_comparison.csv', 'feature_spearman_by_bearing.csv', 'reference_reproduction_check.csv', 'signal_rms_ratio_by_bearing.csv']

=== JSON: COMPAT_FINAL_SUMMARY.json
  status = COMPAT_FAIL_MEASURED
  criteria.onset_diff_max_windows = 5
  criteria.detected_set_must_match = True
  checks.detected_set_match = True
  checks.onset_diff_ok = False
  detection[0].bearing = Bearing1_1
  detection[0].a_detected = True
  detection[0].b_detected = True
  detection[0].a_onset = 1462
  detection[0].b_onset = 1462
  detection[0].onset_diff_windows = 0
  detection[0].a_lead_h = 3.725
  detection[0].b_lead_h = 3.725
  detection[0].a_reg_far = 0.0033333333333333335
  detection[0].b_reg_far = 0.0033333333333333335
  detection[0].a_threshold =

In [ ]:
# ================================================================
# Cell 9 — Next-4: 2048(20.48kHz) + LP6000 M0 측정 (측정 전용)
# D-024 필터 사양 고정: Kaiser FIR, 감쇠 60dB, 전이폭 400Hz, 컷오프 -6dB, 지연 0
# fs=20480용으로 계수를 새로 설계 → FILT_KEY 따로 기록
# 쓰기: CACHE_DIR(캐시) + 새 폴더(*_m0_2048_lp6000_v1)만. 원본·동결·2048 데이터셋 쓰기 없음
# COMPAT 판정(COMPAT_FAIL_MEASURED)·기준·동결 파라미터 변경 없음
# ================================================================
import scipy.signal as sps

_req = ["A", "B", "LEARNING_BEARINGS", "m0_features", "m0_detect", "feat_vec", "with_io_retry",
        "CACHE_DIR", "BRIDGE_ROOT", "DATASET_ROOT", "FN_KEY", "FROZEN_HASH_BEFORE", "sha256_file",
        "write_json", "OUTPUT_FS", "OUTPUT_SAMPLES", "FEATURE_NAMES", "CRITERIA"]
_miss = [n for n in _req if n not in globals()]
if _miss:
    raise RuntimeError(f"선행 셀 변수 없음: {_miss} → Cell 1, 2, 2B, 3, 4 순서로 먼저 실행")

N4_VERSION = "M0_2048_LP6000_v1"
COMPAT_REF_DIR = BRIDGE_ROOT / "20261029_110751_m0_2048_compat_v1"
DIAG_REF_DIR   = BRIDGE_ROOT / "20261029_122814_m0_2048_bandwidth_diag_v1"
DIAG_FILT_KEY  = "651fd4063ef8"          # Cell 8(25.6kHz) 필터 키 — A_LP6000 캐시 조회용
N4_DIR = BRIDGE_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S_m0_2048_lp6000_v1")
N4_DIR.mkdir(parents=True, exist_ok=False)

def n4_stop(reason, **extra):
    write_json(N4_DIR / "N4_SUMMARY.json", {"version": N4_VERSION, "status": "N4_BLOCKED",
                                            "reason": reason, **extra})
    raise RuntimeError(f"N4_BLOCKED: {reason}")

print("FN_KEY:", FN_KEY, "(Cell 8 때와 같아야 A_LP6000 캐시를 찾음: 47c36a35f318497d)")

# ── 필터 설계 (fs=20480) ──
N4_CUTOFFS = {"B_LP_NONE": None, "B_LP8000": 8000.0, "B_LP6000": 6000.0}
FIR_RIPPLE_DB, FIR_WIDTH_HZ = 60.0, 400.0
_nt, N4_BETA = sps.kaiserord(FIR_RIPPLE_DB, FIR_WIDTH_HZ / (OUTPUT_FS / 2))
N4_TAPS = _nt + (1 - _nt % 2)
N4_PAD = N4_TAPS // 2
N4_FIR = {c: sps.firwin(N4_TAPS, fc, window=("kaiser", N4_BETA), fs=OUTPUT_FS)
          for c, fc in N4_CUTOFFS.items() if fc}
N4_FILT_KEY = hashlib.sha256(json.dumps({"fs": OUTPUT_FS, "cut": N4_CUTOFFS, "ripple": FIR_RIPPLE_DB,
    "width": FIR_WIDTH_HZ, "numtaps": int(N4_TAPS), "beta": float(N4_BETA)},
    sort_keys=True).encode()).hexdigest()[:12]
_f = np.array([5800.0, 6000.0, 6200.0])
_, _h = sps.freqz(N4_FIR["B_LP6000"], worN=_f, fs=OUTPUT_FS)
_gain = dict(zip(_f.astype(int).tolist(), np.round(20 * np.log10(np.abs(_h) + 1e-300), 2).tolist()))
print(f"FIR(fs=20480): taps={N4_TAPS}, beta={N4_BETA:.3f}, FILT_KEY={N4_FILT_KEY}")
print("LP6000 응답(dB) 5800/6000/6200Hz:", _gain, "← 6000Hz가 약 -6dB이면 정상")

def lowpass4(x, cond):
    if N4_CUTOFFS[cond] is None:
        return x
    y = np.convolve(np.pad(x, N4_PAD, mode="reflect"), N4_FIR[cond], mode="valid")
    assert y.shape == x.shape
    return y

# ── 측정 ──
det_rows, sp_rows, nreg_rows = [], [], []
t0 = time.time()
for b in LEARNING_BEARINGS:
    arr = with_io_retry(np.load, DATASET_ROOT / "LEARNING" / b / "signals.npy")
    if arr.shape != (len(A[b]["paths"]), OUTPUT_SAMPLES) or arr.dtype != np.float32:
        n4_stop(f"{b}: 2048 배열 shape/dtype 이상 {arr.shape} {arr.dtype}")
    da, db = A[b]["det"], B[b]["det"]
    nreg_rows.append({"bearing": b, "A_n_reg": da["n_reg"], "B_n_reg": db["n_reg"],
                      "A_reg_far": da["reg_far"], "B_reg_far": db["reg_far"]})

    # 비교 기준: 25.6kHz + LP6000 (Cell 8 캐시)
    a6p = CACHE_DIR / f"D_{b}_LP6000_{FN_KEY}_{DIAG_FILT_KEY}.npy"
    FA6 = np.load(a6p) if a6p.is_file() else None
    if FA6 is not None and FA6.shape != (arr.shape[0], 9):
        FA6 = None
    dA6 = m0_detect(FA6) if FA6 is not None else None
    if dA6 is None:
        print(f"⚠ {b}: A_LP6000 캐시 없음 → 해당 비교는 빈칸")
    else:
        det_rows.append({"bearing": b, "cond": "A_LP6000(25.6k)", "detected": dA6["detected"],
                         "onset": dA6["onset_index"], "lead_h": dA6["lead_hours"],
                         "reg_far": dA6["reg_far"], "n_reg": dA6["n_reg"], "threshold": dA6["threshold"]})

    for cond in N4_CUTOFFS:
        fp = CACHE_DIR / f"N4_{b}_{cond}_{FN_KEY}_{N4_FILT_KEY}.npy"
        if fp.is_file() and np.load(fp).shape == (arr.shape[0], 9):
            F = np.load(fp)
        else:
            F = np.zeros((arr.shape[0], 9))
            for i in range(arr.shape[0]):
                x = arr[i].astype(np.float64)
                F[i] = feat_vec(m0_features(lowpass4(x, cond), OUTPUT_FS))[1]
            tmp = fp.with_name(fp.stem + "_tmp.npy"); np.save(tmp, F); os.replace(tmp, fp)

        if cond == "B_LP_NONE" and not np.allclose(F, B[b]["F"], rtol=1e-12, atol=1e-12):
            n4_stop(f"{b}: 필터 없음 조건이 Cell 4 특징과 불일치 — 파이프라인 점검 필요")

        d = m0_detect(F)
        det_rows.append({"bearing": b, "cond": cond, "detected": d["detected"],
                         "onset": d["onset_index"], "lead_h": d["lead_hours"],
                         "reg_far": d["reg_far"], "n_reg": d["n_reg"], "threshold": d["threshold"],
                         "onset_diff_vs_A_LP6000": (d["onset_index"] - dA6["onset_index"])
                             if (dA6 is not None and d["detected"] and dA6["detected"]) else None})
        if FA6 is not None:
            for j, name in enumerate(FEATURE_NAMES):
                r = spearmanr(FA6[:, j], F[:, j]).correlation
                sp_rows.append({"bearing": b, "cond": cond, "feature": name,
                                "spearman_vs_A_LP6000": None if np.isnan(r) else float(r)})
    print(f"[N4] {b} 완료 (누적 {(time.time()-t0)/60:.1f}분)")
    del arr

det4, sp4, nreg4 = pd.DataFrame(det_rows), pd.DataFrame(sp_rows), pd.DataFrame(nreg_rows)
det4.to_csv(N4_DIR / "n4_detection.csv", index=False, encoding="utf-8-sig")
sp4.to_csv(N4_DIR / "n4_spearman_vs_A_LP6000.csv", index=False, encoding="utf-8-sig")
nreg4.to_csv(N4_DIR / "n4_nreg_check_A_vs_B.csv", index=False, encoding="utf-8-sig")

# ── 출력 ──
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
order = ["A_LP6000(25.6k)", "B_LP_NONE", "B_LP8000", "B_LP6000"]
print("\n=== n_reg / reg_FAR: 원본 vs 2048 (reg_FAR 동일 원인 확인) ===")
print(nreg4.to_string(index=False))
for col in ["onset", "lead_h", "reg_far"]:
    print(f"\n=== {col} ===")
    print(det4.pivot(index="cond", columns="bearing", values=col).reindex(order).round(4).to_string())
agg = det4.groupby("cond").agg(detected=("detected", "sum"), mean_lead_h=("lead_h", "mean"),
                               mean_reg_far=("reg_far", "mean")).reindex(order)
agg["mean_reg_far_%"] = (agg["mean_reg_far"] * 100).round(3)
print("\n=== 조건별 요약 ==="); print(agg[["detected", "mean_lead_h", "mean_reg_far_%"]].round(4).to_string())
if len(sp4):
    print("\n=== 특징별 최소 Spearman vs A_LP6000 (6베어링 중 최소) ===")
    print(sp4.groupby(["feature", "cond"])["spearman_vs_A_LP6000"].min().unstack().round(3).to_string())

# ── 참고 판정 (사전 고정: D-021과 같은 기준, B_LP6000 vs A_LP6000) ──
ref_check = None
b6 = det4[det4["cond"] == "B_LP6000"].set_index("bearing")
a6 = det4[det4["cond"] == "A_LP6000(25.6k)"].set_index("bearing")
if len(a6) == 6:
    s6 = sp4[sp4["cond"] == "B_LP6000"]["spearman_vs_A_LP6000"]
    ref_check = {
        "detected_set_match": bool((b6["detected"] == a6.loc[b6.index, "detected"]).all()),
        "onset_diff_ok": bool(b6["onset_diff_vs_A_LP6000"].dropna().abs().le(CRITERIA["onset_diff_max_windows"]).all()),
        "spearman_ok": bool(s6.notna().all() and (s6 >= CRITERIA["feature_spearman_min"]).all()),
        "spearman_below_0_95": int((s6 < CRITERIA["feature_spearman_min"]).sum()),
    }
    ref_check["status"] = ("N4_REF_MATCH" if all(ref_check[k] for k in
                           ["detected_set_match", "onset_diff_ok", "spearman_ok"]) else "N4_REF_MISMATCH_MEASURED")
print("\n참고 판정(B_LP6000 vs A_LP6000):", ref_check)

FROZEN_HASH_AFTER9 = {p: sha256_file(p) for p in FROZEN_HASH_BEFORE}
frozen_ok9 = FROZEN_HASH_AFTER9 == FROZEN_HASH_BEFORE
write_json(N4_DIR / "N4_SUMMARY.json", {
    "version": N4_VERSION, "status": "N4_COMPLETE",
    "note": "측정 전용 — COMPAT 판정·기준·동결 파라미터 변경 없음. 참고 판정은 대표 결정용 정보",
    "compat_ref": str(COMPAT_REF_DIR.name), "diag_ref": str(DIAG_REF_DIR.name),
    "filter": {"type": "kaiser FIR zero-phase (reflect pad, valid conv)", "fs": OUTPUT_FS,
               "numtaps": int(N4_TAPS), "beta": float(N4_BETA), "ripple_db": FIR_RIPPLE_DB,
               "width_hz": FIR_WIDTH_HZ, "cutoff_definition": "-6dB", "cutoffs": N4_CUTOFFS,
               "filt_key": N4_FILT_KEY, "lp6000_gain_db": _gain},
    "fn_key": FN_KEY, "reference_check_B_LP6000_vs_A_LP6000": ref_check,
    "nreg_check": nreg4.to_dict("records"), "detection": det4.to_dict("records"),
    "frozen_files_unchanged": frozen_ok9, "data_scope": "LEARNING 6 bearings only",
    "threshold_or_params_modified": False, "training_performed": False, "m0_reimplemented": False,
    "interpretation_limit": "FEMTO 20.48kHz 결과. 현장 26.667kHz 성능으로 해석 금지",
})
print("\n" + "=" * 60)
print("STATUS: N4_COMPLETE | 동결 파일 불변:", "✅" if frozen_ok9 else "❌")
print("폴더:", N4_DIR)


FN_KEY: 47c36a35f318497d (Cell 8 때와 같아야 A_LP6000 캐시를 찾음: 47c36a35f318497d)
FIR(fs=20480): taps=187, beta=5.653, FILT_KEY=7e32cc3e140c
LP6000 응답(dB) 5800/6000/6200Hz: {5800: -0.01, 6000: -6.02, 6200: -60.96} ← 6000Hz가 약 -6dB이면 정상
[N4] Bearing1_1 완료 (누적 0.4분)
[N4] Bearing1_2 완료 (누적 0.5분)
[N4] Bearing2_1 완료 (누적 0.6분)
[N4] Bearing2_2 완료 (누적 0.8분)
[N4] Bearing3_1 완료 (누적 0.8분)
[N4] Bearing3_2 완료 (누적 1.1분)

=== n_reg / reg_FAR: 원본 vs 2048 (reg_FAR 동일 원인 확인) ===
   bearing  A_n_reg  B_n_reg  A_reg_far  B_reg_far
Bearing1_1      600      600   0.003333   0.003333
Bearing1_2      217      217   0.000000   0.000000
Bearing2_1      227      227   0.008811   0.008811
Bearing2_2      199      199   0.005025   0.005025
Bearing3_1      128      128   0.007812   0.007812
Bearing3_2      409      409   0.002445   0.002445

=== onset ===
bearing          Bearing1_1  Bearing1_2  Bearing2_1  Bearing2_2  Bearing3_1  Bearing3_2
cond                                                                             